# Code-Graph Ontology — Tier 0 Formalization Spec

**Status:** Draft · **Date:** 2026-06-04 · **Crate:** `spur-graph` · **Layer:** graph extraction / schema

## Why this spec

SPUR's graph index is *already* a semantic-triple store: every `edges` row is `(source_stable_id, relation, target_stable_id)` = **(Subject, Predicate, Object)**, carrying provenance (`confidence_score`, `bind_method`). DuckPGQ and the analyst DB sit directly on this shape — the store is RDF-ready.

**What is missing is the predicate — and, one level up, the *ontology* that gives every predicate a single, language-independent meaning.** A vocabulary enum is not an ontology.

This spec defines **Tier 0**: the formal ontology layer (the T-Box) that MUST land before any predicate enrichment (Tier 1+), so that a predicate like `implements` means exactly one thing whether the Subject is Rust, C++, TypeScript, or Python.

## TL;DR

- `RelationKind` already names richer predicates (`Implements`, `Defines`, `Uses`, `Extends`) — **but they emit zero facts.** Declared ≠ realized.
- Tier 0 = (1) formal **predicate signatures** (domain/range), (2) **disambiguation** of overloaded predicates (`calls` → calls / constructs / reads_field / writes_field; `contains` → +defines), (3) a **cross-language realization contract** (which language realizes which predicate, or marks it `—`), (4) **inverse / transitivity / cardinality** metadata, (5) a **gate test** that enforces all of the above.
- Decision forced up front: a **small universal predicate core + explicit "not-realizable" markers**, mirroring the existing node-kind coverage matrix.

## 1. The evidence — a half-built ontology

`RelationKind` (`crates/spur-graph/src/schema.rs:282`) declares **10** predicates. The live graph emits only **6** (5 in `edges`, plus `touches` in `temporal_edges`). Four are **declared but unbound** — named in the type system, realized by no extraction rule.

```mermaid
flowchart LR
  subgraph TBox["T-Box · RelationKind enum (declared vocabulary)"]
    direction TB
    Imports; Calls; Contains; References; Links
    Implements; Defines; Uses; Extends; Touches
  end
  subgraph ABox["A-Box · facts actually emitted"]
    direction TB
    a1["contains · 51,607"]
    a2["calls · 26,319"]
    a3["imports · 4,940"]
    a4["references · 110"]
    a5["links · 28"]
    a6["touches · temporal_edges"]
  end
  Imports --> a3
  Calls --> a2
  Contains --> a1
  References --> a4
  Links --> a5
  Touches --> a6
  Implements -.->|0 facts| VOID["∅ no extraction rule"]
  Defines -.->|0 facts| VOID
  Uses -.->|0 facts| VOID
  Extends -.->|0 facts| VOID
  style VOID fill:#fde,stroke:#c33
  style Implements fill:#fee
  style Defines fill:#fee
  style Uses fill:#fee
  style Extends fill:#fee
```

The four red predicates are the symptom: binding them ad-hoc, per language, would let `implements` mean "Rust impl" in one file and nothing in another — the exact opposite of what a knowledge graph is for.

## 2. Empirical baseline (live graph, 2026-06-04)

Pulled via `spur-analyst` against `.spur/analyst.duckdb`.

**Predicates actually emitted (`edges.relation`):**

| Predicate | Count | Semantics today | Problem |
|---|---|---|---|
| `contains` | 51,607 | structural nesting | overloaded: module⊃fn, struct⊃field, impl⊃method, enum⊃variant |
| `calls` | 26,319 | invocation | overloaded: + field access (4,113) + construction (121) |
| `imports` | 4,940 | `use` resolution | ok |
| `references` | 110 | HOF function-passing only | very narrow |
| `links` | 28 | markdown links | ok |

**Resolution qualifier (`edge_kind`) and binding provenance (`bind_method`) — separate axes, not predicates:**

| edge_kind | n |  | bind_method | n |
|---|---|---|---|---|
| references_other | 56,575 |  | (exact / structural) | 63,541 |
| calls | 26,232 |  | singleton | 11,158 |
| references_hof | 110 |  | scope_match | 5,723 |
| calls_dyn | 87 |  | macro_body_singleton | 1,864 |
|  |  |  | fqn | 718 |

**Unresolved edges (no resolved Object):** `calls` 84,286 · `references_other` 10,382 — the dynamic-dispatch / macro tail, ~59% of all call labels. These are where the inference tier (§12) earns its keep.

```mermaid
pie showData title edges.relation distribution
  "contains" : 51607
  "calls" : 26319
  "imports" : 4940
  "references" : 110
  "links" : 28
```

## 3. The ontology stack

Tier 0 is the bottom band. Every tier above it depends on its uniformity guarantees.

```mermaid
flowchart TB
  subgraph T0["TIER 0 · Ontology (this spec)"]
    direction TB
    classes["Classes / NodeKinds — 18, language-agnostic"]
    preds["Predicates + domain/range signatures (universal core)"]
    axes["Edge qualifier axes — relation · edge_kind · bind_method"]
    meta["Predicate metadata — inverse · transitive · cardinality"]
    realize["Per-language realization contract — Rust / C++ / TS / Tsx / Python / Markdown"]
    gate["Gate test — completeness + uniformity"]
  end
  subgraph T1["TIER 1 · AST-derivable facts"]
    t1a["implements · constructs · reads/writes_field · returns_type · field_of_type · derives"]
  end
  subgraph T2["TIER 2 · cross-artifact fusion (join on stable_symbol_id)"]
    t2a["tests · documented_by · co_changed_with · fixes · implements_plan_task · reviewed_by"]
  end
  subgraph T3["TIER 3 · inferred (confidence-weighted)"]
    t3a["conceptually_similar_to · depends_on · belongs_to_subsystem · may_depend_on (dyn)"]
  end
  T0 --> T1 --> T2 --> T3
  realize -. enforces uniform meaning .-> t1a
  gate -. blocks drift .-> t1a
  style T0 fill:#e8f0ff,stroke:#36c
```

## 4. T-Box · Classes (NodeKind)

The class vocabulary already exists and is language-agnostic (`schema.rs::NodeKind`, 18 variants). Tier 0 only formalizes the **family grouping**, so predicate signatures can reference families instead of enumerating 18 kinds.

```mermaid
flowchart TB
  Symbol --> Container & Callable & TypeDecl & Member & Doc & Artifact
  Container --> File & Module & Impl
  Callable --> Function & Method & Macro
  TypeDecl --> Struct & Enum & Class & Interface & Trait & TypeAlias
  Member --> Field & EnumVariant & Constant
  Doc --> Section & McpTool
  Artifact --> Commit
  style Symbol fill:#e8f0ff,stroke:#36c
```

Family names (`Container`, `Callable`, `TypeDecl`, `Member`, `Doc`, `Artifact`) are **new ontology sugar** for writing compact domain/range constraints. They do not change `NodeKind`.

## 5. T-Box · Predicate catalogue (universal core)

Every predicate gets a **domain** (legal Subject family), a **range** (legal Object family), a **status**, and the **axis** it lives on. Status legend: `✓` emitted today · `◐` declared-but-unbound · `＋` new in Tier 0.

| Predicate | Domain → Range | Status | Notes |
|---|---|---|---|
| `contains` | Container → Symbol | ✓ | keep as the lexical/structural spine |
| `defines` | Container / TypeDecl → Member / Callable | ＋ (split from `contains`) | struct→field, impl→method, enum→variant, trait→method |
| `calls` | Callable → Callable | ✓ (narrowed) | invocation ONLY |
| `constructs` | Callable → TypeDecl | ＋ (split from `calls`) | `X::new`, `X { .. }` |
| `reads_field` | Callable → Field | ＋ (split from `calls`) | field read |
| `writes_field` | Callable → Field | ＋ (split from `calls`) | field write / assign |
| `implements` | Impl / Struct / Class / Enum → Trait / Interface | ◐ → ＋ | the impl / inherit contract |
| `extends` | Trait / Interface / Class → Trait / Interface / Class | ◐ → ＋ | supertrait / base class / TS `extends` |
| `imports` | Symbol → Symbol | ✓ | `use` / `import` |
| `uses` | Callable → TypeDecl | ◐ → ＋ | type referenced in signature/body (non-call) |
| `returns_type` | Callable → TypeDecl | ＋ | return type |
| `accepts_type` | Callable → TypeDecl | ＋ | parameter type |
| `field_of_type` | Field → TypeDecl | ＋ | declared field type |
| `derives` | TypeDecl → Trait | ＋ | `#[derive(..)]` / decorator base |
| `annotated_with` | Symbol → Symbol | ＋ | attribute / decorator / attr-macro |
| `references` | Callable → Callable | ✓ | HOF function-value passing |
| `links` | Section → Symbol | ✓ | markdown link |
| `touches` | Commit → Symbol | ✓ | temporal (separate table) |

`defines` vs `contains`: `contains` stays the lexical-nesting spine (file⊃module⊃…); `defines` is the **semantic** "declares this member" used by impact queries. Both can hold on the same pair; a consumer picks the axis it needs.

## 6. T-Box · Domain → Predicate → Range (signatures)

Domain/range constraints are what stop a predicate from meaning four things. A binder that emits an edge violating its signature is a **gate-test failure at build time**, not a silent row in the parquet.

```mermaid
flowchart LR
  Callable(("Callable"))
  TypeDecl(("TypeDecl"))
  Trait(("Trait / Interface"))
  Field(("Field"))
  Impl(("Impl / Struct / Class"))
  Container(("Container"))
  Member(("Member"))

  Callable -->|calls| Callable
  Callable -->|constructs| TypeDecl
  Callable -->|"reads_field / writes_field"| Field
  Callable -->|"returns_type / accepts_type / uses"| TypeDecl
  Impl -->|implements| Trait
  Trait -->|extends| Trait
  TypeDecl -->|derives| Trait
  Field -->|field_of_type| TypeDecl
  Container -->|"defines / contains"| Member
```

## 7. The three orthogonal edge axes

An edge is not just a predicate. SPUR already stores three independent dimensions; Tier 0 makes them explicit so they stop blurring into each other.

```mermaid
flowchart TB
  edge["one edge"]
  edge --> A["AXIS 1 · relation — WHAT the relationship means<br/>calls / implements / defines / constructs / …"]
  edge --> B["AXIS 2 · edge_kind — HOW it dispatches/resolves<br/>calls · calls_dyn · references_hof · references_other"]
  edge --> C["AXIS 3 · bind_method — HOW CERTAIN the Object binding is<br/>fqn &gt; scope_match &gt; singleton &gt; macro_body_singleton"]
  A -.-> Q["a query filters on the axis it cares about:<br/>'implements, only fqn-certain' = AXIS1=implements ∧ AXIS3=fqn"]
  B -.-> Q
  C -.-> Q
  style A fill:#e8f0ff,stroke:#36c
```

**Tier-0 contract:** `relation` is the predicate; `edge_kind` and `bind_method` are qualifiers, never a substitute for the predicate. Today `references_other` is doing duty as a pseudo-predicate for `contains`/field-access — that conflation ends here.

## 8. Disambiguation MUST precede binding

Overloaded predicates are split **in the T-Box first**, then realized per language. Splitting *after* per-language binding means cleaning the conflation up six times.

```mermaid
flowchart LR
  subgraph Before["TODAY — one predicate, four meanings"]
    f1["fn foo"] -->|calls| g1["fn bar"]
    f1 -->|calls| s1["Cfg::new"]
    f1 -->|calls| fl1["field .count"]
  end
  subgraph After["TIER 0 — distinct predicates"]
    f2["fn foo"] -->|calls| g2["fn bar"]
    f2 -->|constructs| s2["struct Cfg"]
    f2 -->|reads_field| fl2["field .count"]
  end
  Before ==>|split at schema.rs| After
  style After fill:#eefbe8,stroke:#3a3
```

The tree-sitter capture already distinguishes these shapes (`field_expression` vs `call_expression` vs `Type::new` scoped-id) — today they all collapse to `RelationKind::Calls` in `languages.rs`. The split is a **classifier on the existing capture**, not new parsing.

## 9. Cross-language realization contract

The decisive Tier-0 artifact. One universal predicate, realized differently per language — or explicitly **not realizable** (`—`), exactly like the node-kind coverage matrix already in `queries/README.md`.

**Example — how each language realizes `implements`:**

```mermaid
flowchart TB
  P["universal predicate :: implements<br/>domain = Impl/Struct/Class/Enum · range = Trait/Interface"]
  P --> R["Rust — impl Trait for Type<br/>capture: impl_item .trait"]
  P --> C["C++ — class T : public Base<br/>capture: base_class_clause"]
  P --> TS["TypeScript — class C implements I<br/>capture: implements_clause"]
  P --> PY["Python — class C(Proto) / ABC<br/>capture: argument_list base — ◐ heuristic"]
  P --> MD["Markdown — not realizable"]
  style MD fill:#eee,stroke:#999
  style PY fill:#fff6e0
  style P fill:#e8f0ff,stroke:#36c
```

**Predicate × language realization matrix** (`Y` realized · `—` n/a · `◐` heuristic · `TODO` gap):

| Predicate | Rust | C++ | TS | Tsx | Python | Markdown |
|---|---|---|---|---|---|---|
| contains | Y | Y | Y | Y | Y | Y |
| defines | Y | Y | Y | Y | Y | — |
| calls | Y | Y | Y | Y | Y | — |
| constructs | Y | Y | Y | Y | Y | — |
| reads_field / writes_field | Y | Y | Y | Y | Y | — |
| implements | Y | Y | Y | Y | ◐ | — |
| extends | Y (supertrait) | Y (base) | Y | Y | ◐ | — |
| imports | Y | Y | Y | Y | Y | Y (links) |
| returns_type / accepts_type | Y | Y | Y | Y | ◐ (hints) | — |
| field_of_type | Y | Y | Y | Y | ◐ | — |
| derives / annotated_with | Y | TODO | Y (decorator) | Y | Y (decorator) | — |
| references (HOF) | Y | TODO | Y | Y | TODO | — |
| links | — | — | — | — | — | Y |

`◐` = realizable only heuristically (Python's dynamic typing) → emit with reduced `bind_method` / confidence, **never** as `fqn`-exact.

## 10. The binding pipeline (where Tier 0 plugs in)

```mermaid
flowchart LR
  src["source file"] --> ts["tree-sitter parse"]
  ts --> q["spur-edges.scm capture<br/>@call.name @import.name<br/>@impl.trait (new) @ctor (new) …"]
  q --> cls["capture → predicate classifier<br/>(languages.rs)"]
  cls --> sig{"domain/range<br/>signature check"}
  sig -->|legal| edge["GraphEdge { relation, edge_kind, bind_method, confidence }"]
  sig -->|illegal| fail["GATE-TEST FAILURE<br/>(build-time, not a runtime row)"]
  edge --> store["edges parquet → analyst.duckdb → DuckPGQ"]
  subgraph Tier0["Tier-0 owns"]
    q; cls; sig
  end
  style Tier0 fill:#e8f0ff,stroke:#36c
  style fail fill:#fde,stroke:#c33
```

Today the classifier (`languages.rs` capture dispatch) maps `call.name→Calls`, `import.name→Imports`, `reference.name→References`, structural→`Contains`. Tier 0 extends it to the full predicate set **and** inserts the signature check the gate test reads.

## 11. Predicate metadata — inverse · transitive · cardinality

Required for sound `MATCH` reasoning and cheap reverse lookups.

| Predicate | Inverse | Transitive | Cardinality | Symmetric |
|---|---|---|---|---|
| contains | contained_by | yes | 1 parent : N | no |
| defines | defined_in | no | 1 : N | no |
| calls | called_by | no | N : N | no |
| constructs | constructed_by | no | N : N | no |
| implements | implemented_by | no | N : N | no |
| extends | extended_by | **yes** | N : N | no |
| imports | imported_by | no | N : N | no |
| returns_type / accepts_type | used_in | no | N : N | no |
| co_changed_with (T2) | *self* | no | N : N | **yes** |

```mermaid
flowchart LR
  A["trait A"] -->|extends| B["trait B"] -->|extends| C["trait C"]
  A -. "entailed: extends* (transitive closure)" .-> C
  style C stroke-dasharray: 4 4
```

Transitive closures (`contains*`, `extends*`) are **derived, not stored** — computed on demand via recursive CTE / DuckPGQ `{1,N}`. Tier 0 only *declares* transitivity so the query layer knows the closure is sound.

## 12. Entailment — predicates make inference sound

Typed predicates + metadata let the inference tier (Tier 3) derive edges that are not syntactically present — the principled answer to the 59% unresolved dynamic calls.

```mermaid
flowchart TB
  f1["B::run calls dyn 'm'<br/>edge_kind=calls_dyn · Object unresolved"]
  f2["A implements Trait T"]
  f3["T defines method m"]
  f1 --> rule{{"inference rule"}}
  f2 --> rule
  f3 --> rule
  rule ==>|derive| out["B::run may_depend_on A::m<br/>relation=may_depend_on · bind_method=inferred · confidence=Heuristic"]
  style out fill:#eefbe8,stroke:#3a3
```

Without `implements` + `defines`, this rule has no premises to fire on. **The ontology is the precondition for inference**, not an optional nicety.

## 13. Tier-0 deliverables

1. **`schema.rs` — predicate set.** Split `Calls` → `{Calls, Constructs, ReadsField, WritesField}`; add `Defines, ReturnsType, AcceptsType, FieldOfType, Derives, AnnotatedWith`; bind the orphans `Implements, Extends, Uses`. Add a `PredicateSig { domain: &[NodeFamily], range: &[NodeFamily], inverse, transitive, cardinality }` table.
2. **`queries/*/spur-edges.scm` — captures.** New patterns per language: `impl_item .trait` (Rust), `base_class_clause` (C++), `implements_clause` / `extends_clause` (TS), `field_expression` lvalue vs rvalue, `Type::new` / struct literal for `constructs`, attribute / decorator for `derives` / `annotated_with`.
3. **`languages.rs` — classifier + matrix.** Extend the capture dispatch to the full predicate set; add a `PREDICATE_REALIZATION` matrix (lang × predicate → Y / — / ◐ / TODO).
4. **Gate test** (extends the `languages.rs` contract test). Enforce: every emitted predicate has a signature; every signature's domain/range references valid families; every `(lang, predicate)=Y` has ≥1 capture that produces it; every capture maps to a predicate; the matrix has no unreviewed `TODO`.
5. **`queries/README.md` — the relation coverage matrix** (mirror of the existing node matrix).

```mermaid
flowchart LR
  d1["schema.rs<br/>predicate set + signatures"] --> d3["languages.rs<br/>classifier + matrix"]
  d2["*.scm captures"] --> d3
  d3 --> d4["gate test enforces<br/>completeness + uniformity"]
  d4 --> d5["README relation matrix<br/>(human contract)"]
  style d4 fill:#e8f0ff,stroke:#36c
```

## 14. Sequencing

Tier 0 ships **before** any Tier-1 enrichment. Within Tier 0, the schema + signature + gate-test skeleton lands first (it can pass with the *current* 5 predicates), then each predicate is bound language-by-language behind the gate.

```mermaid
gantt
  dateFormat YYYY-MM-DD
  axisFormat %m-%d
  title Tier-0 rollout (illustrative ordering, not commitments)
  section Schema
  Predicate enum + PredicateSig table       :a1, 2026-06-05, 2d
  Gate-test skeleton (green on 5 preds)     :a2, after a1, 1d
  section Disambiguation
  Split calls into calls/ctor/r-w_field     :b1, after a2, 2d
  Split contains into +defines              :b2, after a2, 1d
  section Type + contract predicates
  implements + extends (all langs)          :c1, after b1, 3d
  returns/accepts/field_of_type             :c2, after c1, 3d
  derives / annotated_with                  :c3, after c1, 2d
  section Contract
  Relation coverage matrix + README         :d1, after c3, 1d
```

## 15. Open decisions

1. **`defines` vs `contains` co-existence** — emit both (richer, ~2× member edges) or replace `contains` for member pairs (smaller, but breaks existing `contains` consumers like `v_doc_tree` and `lance_sections`, which filter `relation=Contains`)? *Leaning: emit both; `contains` stays the lexical spine.*
2. **Python `◐` predicates** — bind heuristically (Protocol/ABC bases, type hints) at reduced confidence, or mark `TODO` and skip until a type-inference pass exists? *Leaning: heuristic + `bind_method=scope_match`, never `fqn`.*
3. **Universal-core ceiling** — how large can the core grow before it stops being "universal"? Proposed rule: a predicate joins the core only if **≥2 language families realize it `Y`**; single-language concepts stay as `edge_kind` qualifiers.
4. **`uses` scope** — "any type mentioned in a body" risks exploding edge count (every local's type). Bound it to signature positions + struct literals, or gate behind a confidence floor?
5. **Inverse storage** — store inverse edges, or derive at query time via `edges_by_dst` (already exists for `calls`)? *Leaning: derive; do not double the table.*

## Cross-references

- `crates/spur-graph/src/schema.rs` — `NodeKind` / `RelationKind` / `GraphEdgeKind`
- `crates/spur-graph/src/extract/languages.rs` — capture→predicate classifier + the contract gate test
- `crates/spur-graph/queries/*/spur-edges.scm` — per-language edge captures
- `crates/spur-graph/queries/README.md` — the node-kind coverage matrix (the model to mirror)
- Prior art: `2026-05-05-spur-graph-engine-design.md`, `2026-05-11-tree-sitter-code-mentions-design.md`, `2026-05-20-code-as-memory-phase-1-design.md`

## Appendix · reproduce the empirical baseline

The §2 numbers came from `spur-analyst`. This read-only cell reproduces them against the live analyst DB (left unexecuted in the committed spec — run it locally to refresh).

In [ ]:
import duckdb
con = duckdb.connect("/Volumes/Projects/spur/.spur/analyst.duckdb", read_only=True)

# Predicates actually emitted, by count:
print(con.sql("SELECT relation AS predicate, count(*) AS n FROM edges GROUP BY 1 ORDER BY n DESC").df())

# Predicates DECLARED in RelationKind but never emitted (the orphans):
declared = {"imports","calls","contains","implements","defines",
            "references","uses","extends","links","touches"}
emitted = set(con.sql("SELECT DISTINCT relation FROM edges").df()["relation"])
print("declared-but-unbound:", sorted(declared - emitted - {"touches"}))